# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/improve/model-accuracy/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Compare Logistic Regression, Decision Tree, Random Forest, and LightGBM, same feature set, same split, ranked by Precision@50.

**Why:** Earlier testing surfaced two real bugs, a GA4-availability filter that silently dropped most of the panel, and a static `content_updated_date` field that leaked future edit information into a "days since update" feature, both now fixed. It also showed that grouping by `content_hash_id` alone isn't sufficient, pages from the same client move together, so validation must hold out entire clients, matching the reference pipeline's own approach. Trying multiple model families under this corrected, honest split shows whether the earlier RandomForest choice was actually the right one or just the default, useful evidence either way for the capstone write-up.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Design:** Client-level holdout (GroupShuffleSplit on `client_hash_id`, 75/25), averaged across 5 random seeds.

**Why:** A time-based or content-based split both looked valid at first, but testing showed same-client pages share enough underlying similarity that both leaked, Precision@50 hit a suspicious 1.000. Holding out entire clients is the only split where the model has never seen anything from that client, matching the validation method the reference 74% baseline itself used. With only 36 clients total in this warehouse slice, a single split is unstable, a handful of large clients landing in test can swing the result heavily, so the result is reported as a mean and standard deviation across 5 random splits, not a single point estimate.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# Setup: install and authenticate
!pip install -q duckdb huggingface_hub lightgbm


In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    import getpass
    login(token=getpass.getpass("Paste HF token: "))


In [ ]:
# Download dim_content and all 18 monthly fact files, sequentially with retry to avoid HF rate limits
import time
from huggingface_hub import hf_hub_download

dim_content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                     filename="dim_content.parquet")

all_months = [f"2025-{m:02d}" for m in range(1, 13)] + [f"2026-{m:02d}" for m in range(1, 7)]
all_files = []

for m in all_months:
    for attempt in range(3):
        try:
            path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet")
            all_files.append(path)
            print(f"Downloaded {m}")
            time.sleep(2)
            break
        except Exception as e:
            print(f"Retry {attempt+1} for {m}: {e}")
            time.sleep(10)

print(f"\nTotal files downloaded: {len(all_files)} of {len(all_months)}")


In [ ]:
# Build 90-day feature windows / 30-day future label windows per content item, one snapshot per month,
# joined to dim_content metadata. Excludes client_08a6a72ff48e62c0 (synthetic-signature data quality issue,
# see section 4) and only requires gsc_data_available (GA4 columns kept as NaN when not tracked yet, since
# most clients' GA4 history only starts around Feb 2026).
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

query = f"""
WITH content_meta AS (
    SELECT
        content_hash_id, content_type, main_intent, competition_level, provider_used,
        search_volume, competition, cpc, backlinks, category_count, word_count, char_count,
        content_created_date
    FROM read_parquet('{dim_content_path}')
    WHERE is_published = TRUE AND is_deleted = FALSE
),
base AS (
    SELECT
        client_hash_id, content_hash_id, report_date, ga4_data_available,
        gsc_impressions, gsc_clicks, gsc_sum_position,
        ga4_engaged_sessions, ga4_total_engagement_sec,
        sessions_organic, scroll_events
    FROM read_parquet({all_files})
    WHERE gsc_data_available = TRUE
      AND client_hash_id != 'client_08a6a72ff48e62c0'
),
rolled AS (
    SELECT
        *,
        SUM(gsc_impressions) OVER w90 AS impressions_90d,
        SUM(gsc_clicks) OVER w90 AS clicks_90d,
        SUM(gsc_sum_position) OVER w90 AS sum_position_90d,
        SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions ELSE NULL END) OVER w90 AS engaged_sessions_90d,
        SUM(CASE WHEN ga4_data_available THEN ga4_total_engagement_sec ELSE NULL END) OVER w90 AS engagement_sec_90d,
        SUM(CASE WHEN ga4_data_available THEN sessions_organic ELSE NULL END) OVER w90 AS sessions_organic_90d,
        SUM(CASE WHEN ga4_data_available THEN scroll_events ELSE NULL END) OVER w90 AS scroll_events_90d,
        STDDEV(gsc_sum_position / NULLIF(gsc_impressions, 0)) OVER w7 AS position_roll7_std,
        SUM(gsc_impressions) OVER w30 AS impressions_last30_feature,
        SUM(gsc_impressions) OVER wfut AS impressions_next30,
        ROW_NUMBER() OVER (
            PARTITION BY content_hash_id, DATE_TRUNC('month', report_date)
            ORDER BY report_date DESC
        ) AS rn_in_month
    FROM base
    WINDOW
        w90  AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 89 DAYS PRECEDING AND CURRENT ROW),
        w30  AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 29 DAYS PRECEDING AND CURRENT ROW),
        w7   AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW),
        wfut AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 1 DAYS FOLLOWING AND INTERVAL 30 DAYS FOLLOWING)
),
snapshots AS (
    SELECT * FROM rolled WHERE rn_in_month = 1
)
SELECT
    s.*,
    m.content_type, m.main_intent, m.competition_level, m.provider_used,
    m.search_volume, m.competition, m.cpc, m.backlinks, m.category_count,
    m.word_count, m.char_count,
    DATE_DIFF('day', m.content_created_date, s.report_date) AS content_age_days
FROM snapshots s
JOIN content_meta m USING (content_hash_id)
WHERE DATE_DIFF('day', m.content_created_date, s.report_date) >= 90
  AND s.impressions_90d > 0
  AND s.impressions_next30 IS NOT NULL
"""

df = con.execute(query).df()
print(df.shape)


In [ ]:
# Drop snapshot months whose 90-day feature window or 30-day future window falls too close to the
# warehouse's actual data boundaries (2025-01-27 to 2026-06-30) — these produce artificially thin
# windows that manufacture fake trend signal, confirmed by testing (Apr 2025 and May 2026 both showed
# anomalously low impressions_next30 averages compared to neighboring months).
df["report_date"] = pd.to_datetime(df["report_date"])
df = df[(df["report_date"] >= pd.Timestamp("2025-05-01")) & (df["report_date"] <= pd.Timestamp("2026-04-30"))].copy()
print(df.shape)


In [ ]:
# Label: forward-looking decline. impressions_last30_feature (last 30 days INSIDE the 90-day feature
# window) vs impressions_next30 (the 30 days immediately after, 0-day gap). A minimum-impression floor
# of 100 is applied first — below that, single-digit impression swings make the trend percentage
# meaningless (confirmed: without this floor, label rate collapsed to ~13-18% instead of a stable ~25-55%).
eligible = df["impressions_last30_feature"] >= 100

trend_pct = np.where(
    df["impressions_last30_feature"] > 0,
    (df["impressions_next30"] - df["impressions_last30_feature"]) / df["impressions_last30_feature"] * 100,
    np.nan
)
df["trend_pct"] = trend_pct
df["trend_direction"] = np.select(
    [~eligible, df["trend_pct"] > 20, df["trend_pct"] < -20],
    ["insufficient_volume", "up", "down"],
    default="stable"
)
df["is_declining_label"] = (df["trend_direction"] == "down").astype("int8")

model_df = df[df["trend_direction"] != "insufficient_volume"].copy()
print(f"Eligible rows: {len(model_df)} of {len(df)}")
print(model_df.groupby(model_df["report_date"].dt.to_period("M"))["is_declining_label"].agg(["mean", "count"]))


In [ ]:
# Features: GSC performance, engagement (NaN where GA4 untracked), and dim_content metadata.
# NOTE: content_age_days is deliberately EXCLUDED here — see section 4 for why (it was found to be
# confounded with calendar month in this monthly-snapshot design, inflating apparent accuracy without
# reflecting genuine per-page risk).
model_df["ctr_90d"] = model_df["clicks_90d"] / model_df["impressions_90d"] * 100
model_df["avg_position_90d"] = model_df["sum_position_90d"] / model_df["impressions_90d"]

cat_cols = ["content_type", "main_intent", "competition_level", "provider_used"]
for c in cat_cols:
    model_df[c] = model_df[c].fillna("unknown").astype("category")

model_df_encoded = pd.get_dummies(model_df, columns=cat_cols, drop_first=True)

numeric_features = [
    "impressions_90d", "clicks_90d", "ctr_90d", "avg_position_90d",
    "engaged_sessions_90d", "engagement_sec_90d", "sessions_organic_90d", "scroll_events_90d",
    "position_roll7_std", "search_volume", "competition", "cpc",
    "backlinks", "category_count", "word_count", "char_count"
]
dummy_cols = [c for c in model_df_encoded.columns if any(c.startswith(f"{cc}_") for cc in cat_cols)]
features = numeric_features + dummy_cols

print(f"Total features: {len(features)}")
print(f"Rows: {len(model_df_encoded)}")


In [ ]:
# Client-level holdout split (single seed, for the head-to-head model comparison below)
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df_encoded, groups=model_df_encoded["client_hash_id"]))

train = model_df_encoded.iloc[train_idx]
test = model_df_encoded.iloc[test_idx]

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"Overlapping clients: {len(overlap)} (should be 0)")
print(f"Train: {len(train)}, Test: {len(test)}")
print(f"Train label rate: {train['is_declining_label'].mean():.3f}, Test label rate: {test['is_declining_label'].mean():.3f}")


In [ ]:
# Head-to-head: Logistic Regression, Decision Tree, Random Forest, LightGBM — same features, same split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

X_train, y_train = train[features], train["is_declining_label"]
X_test, y_test = test[features], test["is_declining_label"]

def precision_at_50(y_true, scores):
    top50_idx = pd.Series(scores).nlargest(50).index
    return y_true.iloc[top50_idx].mean()

results = {}

# LR and tree models need NaNs filled (GA4 gaps); LightGBM handles NaN natively
X_train_filled = X_train.fillna(0)
X_test_filled = X_test.fillna(0)

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_filled, y_train)
results["Logistic Regression"] = precision_at_50(y_test.reset_index(drop=True), lr.predict_proba(X_test_filled)[:, 1])

dt = DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=42)
dt.fit(X_train_filled, y_train)
results["Decision Tree"] = precision_at_50(y_test.reset_index(drop=True), dt.predict_proba(X_test_filled)[:, 1])

rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train_filled, y_train)
results["Random Forest"] = precision_at_50(y_test.reset_index(drop=True), rf.predict_proba(X_test_filled)[:, 1])

gbm = lgb.LGBMClassifier(n_estimators=300, max_depth=6, num_leaves=31, learning_rate=0.05,
                           is_unbalance=True, random_state=42, verbosity=-1)
gbm.fit(X_train, y_train)
results["LightGBM"] = precision_at_50(y_test.reset_index(drop=True), gbm.predict_proba(X_test)[:, 1])

results_df = pd.DataFrame({"Model": results.keys(), "Precision@50": results.values()})
results_df["Precision@50"] = results_df["Precision@50"].apply(lambda x: f"{x:.3f}")
print(results_df.to_string(index=False))
print("\nWeek 4 baseline for comparison: 0.240")


In [ ]:
# Final, honest evaluation: LightGBM, content_age_days excluded (see section 4), March-April 2026
# excluded (an unusually large, genuine decline spike across many clients — see section 4), averaged
# across 5 random client-holdout splits since only 36 clients exist and any single split is unstable.
seeds = [42, 7, 123, 2024, 99]
scores = []
spike_months = ["2026-03", "2026-04"]

for seed in seeds:
    gss_s = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(gss_s.split(model_df_encoded, groups=model_df_encoded["client_hash_id"]))
    tr, te = model_df_encoded.iloc[tr_idx], model_df_encoded.iloc[te_idx]

    te = te[~te["report_date"].dt.to_period("M").astype(str).isin(spike_months)]
    if len(te) < 50:
        continue

    gbm_s = lgb.LGBMClassifier(n_estimators=300, max_depth=6, num_leaves=31,
                                 learning_rate=0.05, is_unbalance=True, random_state=42, verbosity=-1)
    gbm_s.fit(tr[features], tr["is_declining_label"])
    te = te.copy()
    te["risk"] = gbm_s.predict_proba(te[features])[:, 1]
    p50 = te.nlargest(50, "risk")["is_declining_label"].mean()
    scores.append(p50)
    print(f"Seed {seed}: Precision@50 = {p50:.3f}, test clients = {te['client_hash_id'].nunique()}, test rows = {len(te)}")

print(f"\nMean: {np.mean(scores):.3f}, Std: {np.std(scores):.3f}, Range: {min(scores):.3f}-{max(scores):.3f}")
print("Week 4 baseline: 0.240 | Reference RandomForest (starter CSV): 0.740 | Original Week-5 attempt: 0.560")


**Final results table:**

| Model | Precision@50 |
|---|---|
| Week 4 Baseline (heuristic) | 0.240 |
| Original Week-5 attempt (3 raw features, single-day label) | 0.560 |
| Logistic Regression | 0.560 |
| Decision Tree | 0.920 (unstable — high-variance model, see section 4) |
| Random Forest | 0.260 |
| **LightGBM, single split** | 0.940 (inflated — see section 4) |
| **LightGBM, mean of 5 client-holdout splits, spike months excluded** | **0.788 (σ = 0.124, range 0.580–0.960)** |


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Three real issues were found and fixed during development, not just assumed away:**

1. **A GA4-availability filter silently dropped most of the panel.** Filtering rows to `ga4_data_available = TRUE` looked like a safe correctness fix, but most of the 51 clients' GA4 tracking didn't start until around February 2026, so this filter discarded the majority of otherwise-valid GSC data from earlier months. Fixed by only requiring `gsc_data_available = TRUE`, and keeping GA4-derived features as `NaN` (not zero) when tracking wasn't active yet.

2. **One client (`client_08a6a72ff48e62c0`) shows a data-quality anomaly.** Hundreds of distinct content items belonging to this client report near-identical aggregate 90-day metrics on the same snapshot date (e.g. ~31,161 impressions and ~6.11 average position across many unrelated pages), while the same pages show different, unique numbers on other snapshot dates. This is inconsistent with how independent page traffic behaves and is most likely a synthetic-data generation artifact specific to this pseudonymized release. This client was excluded from modeling.

3. **`content_age_days` was found to be confounded with calendar month.** Because this pipeline takes one snapshot per content item per month, `content_age_days` rises by roughly 30 with every successive snapshot of the same page, regardless of the page's real age. This made `content_age_days` an effective stand-in for "which month is this row from," and a genuine, sustained decline spike across many clients in March–April 2026 meant the model was substantially rewarded for detecting *that specific period* rather than learning transferable per-page risk (confirmed: with this feature included, 78% of the top-50 flagged rows came from April 2026 alone). The feature was removed, and the March–April spike months were excluded from the final evaluation as an anomalous period warranting separate investigation, not folded into the general-case number.

**Validation methodology, and why it matters here specifically:**

Two earlier split designs (time-based, and grouped by `content_hash_id`) both produced a suspicious Precision@50 of 1.000. Investigation showed pages belonging to the same client are similar enough that neither split actually prevented the model from learning client-specific shortcuts. A client-level `GroupShuffleSplit` (no client's data appears in both train and test) removed this, dropping the single-split score to a more believable 0.52–0.94 depending on the random seed. With only 36 total clients in this dataset, any one split is inherently unstable, a few large clients landing in the test set can swing the result by 30+ points. The reported result is therefore the mean and standard deviation across 5 random seeds (**0.788, σ = 0.124**), not a single point estimate, which is the more defensible way to report a result on this small a number of independent groups.

**What this model can and cannot claim:** it ranks pages by decline risk better than the transparent baseline, using signals available at prediction time (traffic, engagement, content metadata), evaluated on clients the model has never seen. It cannot claim to generalize equally well during anomalous periods like the March–April 2026 spike, which showed different dynamics and was deliberately excluded pending further investigation, and its real-world performance should be expected to fall somewhere in the observed 0.58–0.96 range rather than at any single fixed number.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
